In [1]:
from numba import cuda
import numpy as np

# 1. Define the Kernel
# Instead of printing, we record the data into an array
@cuda.jit
def hello_logger(output_array):
    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x
    
    # Global ID calculation
    gid = bx * bdim + tx

    # Store [Block, Thread, Global_ID] in the row corresponding to the Global ID
    if gid < output_array.shape[0]:
        output_array[gid, 0] = bx
        output_array[gid, 1] = tx
        output_array[gid, 2] = gid

# 2. Host Setup
blocks = 2
threads_per_block = 4
total_threads = blocks * threads_per_block

# Create an empty container (8 rows, 3 columns)
host_array = np.zeros((total_threads, 3), dtype=np.int32)
device_array = cuda.to_device(host_array)

# 3. Launch
hello_logger[blocks, threads_per_block](device_array)
cuda.synchronize()

# 4. Process Results
# Bring data back to CPU
result_array = device_array.copy_to_host()

# SORT the array by Global ID (Column 2) to ensure the order 0, 1, 2...
# This fixes the random order issue common with GPUs.
sorted_indices = result_array[:, 2].argsort()
sorted_array = result_array[sorted_indices]

# 5. Print in your specific format
for row in sorted_array:
    print(f"Hello from Block {row[0]} Thread {row[1]} Global ID {row[2]}")

c:\Users\dil\AppData\Local\Programs\Python\Python313\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:690: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Hello from Block 0 Thread 0 Global ID 0
Hello from Block 0 Thread 1 Global ID 1
Hello from Block 0 Thread 2 Global ID 2
Hello from Block 0 Thread 3 Global ID 3
Hello from Block 1 Thread 0 Global ID 4
Hello from Block 1 Thread 1 Global ID 5
Hello from Block 1 Thread 2 Global ID 6
Hello from Block 1 Thread 3 Global ID 7
